# N-Step Methods and Eligibility Traces
## Bridging Monte Carlo and Temporal Difference Learning

**What You'll Learn:**
- N-step TD returns and their spectrum from TD(0) to Monte Carlo
- N-step SARSA and n-step Q-Learning (Tree Backup)
- Eligibility traces: accumulating and replacing
- TD($\lambda$): forward and backward views
- How $\lambda$ and $n$ affect learning speed and stability

**Prerequisites:** TD Learning (Notebook 3), MC Methods (Notebook 2).

**References:**
- Sutton & Barto, *Reinforcement Learning: An Introduction*, Chapters 7 & 12
- Singh & Sutton, 1996 — *Reinforcement Learning with Replacing Eligibility Traces*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2
})

# Color palette
COLORS = ['steelblue', 'coral', 'seagreen', 'goldenrod', 'mediumpurple']

# Constants
GAMMA = 0.99
ALPHA = 0.1
N_EPISODES = 300

print("Setup complete.")

---
## 1. N-Step Returns: The Spectrum from TD(0) to Monte Carlo

The key idea behind n-step methods is that we can **interpolate** between TD(0) (which uses a 1-step return) and Monte Carlo (which uses the full return). The n-step return looks ahead $n$ steps and then bootstraps from the estimated value:

$$\boxed{G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k R_{t+k+1} + \gamma^n V(S_{t+n})}$$

**Spectrum of methods:**

| n | Method | Bias | Variance |
|---|--------|------|----------|
| 1 | TD(0) | High | Low |
| 2 | 2-step TD | ... | ... |
| 4 | 4-step TD | ... | ... |
| $\infty$ | Monte Carlo | None | High |

As $n$ increases, **bias decreases** (less reliance on possibly inaccurate $V$) but **variance increases** (more stochastic transitions contribute).

---
## 2. N-Step TD Prediction and Control

**N-step TD prediction** updates the value function after observing $n$ transitions:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[ G_t^{(n)} - V(S_t) \right]$$

Key implementation details:
- We must **store** the last $n$ transitions (states, rewards) before making an update
- Updates are **delayed** by $n$ steps
- At the end of an episode, we must flush the remaining transitions

**N-step SARSA** extends this to control by using action-values:

$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k R_{t+k+1} + \gamma^n Q(S_{t+n}, A_{t+n})$$

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ G_t^{(n)} - Q(S_t, A_t) \right]$$

---
## 3. Eligibility Traces

Eligibility traces combine **frequency** and **recency** heuristics to assign credit. A state's eligibility trace reflects how recently and frequently it has been visited.

**Accumulating traces:**

$$e_t(s) = \gamma \lambda \, e_{t-1}(s) + \mathbf{1}(S_t = s)$$

Every visit to $s$ adds 1, and the trace decays by $\gamma\lambda$ per step. This can grow unboundedly for frequently visited states.

**Replacing traces:**

$$e_t(s) = \max\!\big(\gamma \lambda \, e_{t-1}(s),\; \mathbf{1}(S_t = s)\big)$$

Replacing traces cap the trace at 1 upon each visit, preventing unbounded growth and often improving performance.

---
## 4. TD($\lambda$): Forward and Backward Views

**Forward view** — the $\lambda$-return:

$$\boxed{G_t^\lambda = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}}$$

This is a geometrically weighted average of **all** n-step returns. When $\lambda = 0$, we recover TD(0); when $\lambda = 1$, we recover the Monte Carlo return.

**Backward view** — uses eligibility traces for an efficient, online, incremental implementation:

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$
$$e_t(s) = \gamma \lambda \, e_{t-1}(s) + \mathbf{1}(S_t = s)$$
$$V(s) \leftarrow V(s) + \alpha \, \delta_t \, e_t(s) \quad \forall s$$

**Equivalence Theorem:** The forward and backward views produce identical updates in the **offline** (end-of-episode) case, and are approximately equivalent online for small $\alpha$.

---
## 5. Environment: 19-State Random Walk

In [ ]:
class RandomWalkEnv:
    """19-state random walk environment.

    States 1-19 are non-terminal. State 0 (left) and state 20 (right)
    are terminal. The agent starts in the center (state 10) and takes
    left (-1) or right (+1) steps.

    Args:
        n_states: Number of non-terminal states (default 19).

    Returns:
        An environment with reset() and step(action) methods.
    """

    def __init__(self, n_states: int = 19):
        self.n_states = n_states
        self.left_terminal = 0
        self.right_terminal = n_states + 1
        self.start_state = n_states // 2 + 1  # center state (10)
        self.state = self.start_state
        self.n_actions = 2  # 0 = left, 1 = right

    def reset(self) -> int:
        """Reset the environment to the starting state.

        Returns:
            The starting state.
        """
        self.state = self.start_state
        return self.state

    def step(self, action: int) -> Tuple[int, float, bool]:
        """Take a step in the environment.

        Args:
            action: 0 for left, 1 for right.

        Returns:
            Tuple of (next_state, reward, done).
        """
        direction = -1 if action == 0 else 1
        self.state += direction

        if self.state == self.right_terminal:
            return self.state, 1.0, True
        elif self.state == self.left_terminal:
            return self.state, -1.0, True
        else:
            return self.state, 0.0, False

    def true_values(self, gamma: float = 1.0) -> np.ndarray:
        """Compute true state values analytically.

        For the random walk with equiprobable random policy (equal probability
        of going left or right), the true values form a linear gradient
        when gamma=1.0. For gamma<1.0, we solve the Bellman equation directly.

        Args:
            gamma: Discount factor.

        Returns:
            Array of true values for states 0 to n_states+1.
        """
        n = self.n_states
        # Solve the system: V(s) = 0.5 * [gamma * V(s-1) + 0] + 0.5 * [gamma * V(s+1) + 0]
        # with boundary conditions: V(0) contributed as r=-1, V(n+1) contributed as r=+1
        # Set up linear system: for each state s in {1,...,n}
        # V(s) = 0.5 * (0 + gamma * V(s-1)) + 0.5 * (0 + gamma * V(s+1))
        # with V(s-1) for s=1 contributing reward -1, V(s+1) for s=n contributing reward +1

        # Build matrix A and vector b: A @ V = b
        A = np.zeros((n, n))
        b = np.zeros(n)

        for i in range(n):
            s = i + 1  # state number (1-indexed)
            A[i, i] = 1.0
            # Left neighbor
            if s - 1 == self.left_terminal:
                # Reward -1, transition to terminal
                b[i] += 0.5 * (-1.0)
            else:
                A[i, i - 1] = -0.5 * gamma

            # Right neighbor
            if s + 1 == self.right_terminal:
                # Reward +1, transition to terminal
                b[i] += 0.5 * (1.0)
            else:
                A[i, i + 1] = -0.5 * gamma

        V_interior = np.linalg.solve(A, b)
        V = np.zeros(n + 2)
        V[1:n + 1] = V_interior
        return V


# Quick test
env = RandomWalkEnv()
true_v = env.true_values(gamma=1.0)
print(f"Environment: {env.n_states} non-terminal states")
print(f"Start state: {env.start_state}")
print(f"True values (gamma=1.0) — should be linear from -1 to +1:")
print(f"  States 1,5,10,15,19: {true_v[1]:.3f}, {true_v[5]:.3f}, {true_v[10]:.3f}, {true_v[15]:.3f}, {true_v[19]:.3f}")

---
## 6. N-Step Spectrum Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

# Spectrum from TD(0) to MC
n_values = [1, 2, 4, 8, 16, 'inf']
labels = ['n=1\n(TD(0))', 'n=2', 'n=4', 'n=8', 'n=16', 'n=inf\n(MC)']
positions = np.linspace(0, 1, len(n_values))

# Draw the spectrum bar
gradient = np.linspace(0, 1, 256).reshape(1, -1)
ax.imshow(gradient, aspect='auto', cmap='RdYlBu_r', extent=[0, 1, -0.15, 0.15], alpha=0.6)

for pos, label in zip(positions, labels):
    ax.plot(pos, 0, 'ko', markersize=12, zorder=5)
    ax.annotate(label, (pos, 0), textcoords='offset points',
                xytext=(0, 25), ha='center', fontsize=11, fontweight='bold')

# Annotations for bias/variance
ax.annotate('High Bias\nLow Variance', (0, 0), textcoords='offset points',
            xytext=(0, -35), ha='center', fontsize=10, color='steelblue')
ax.annotate('Low Bias\nHigh Variance', (1, 0), textcoords='offset points',
            xytext=(0, -35), ha='center', fontsize=10, color='coral')

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.5, 0.5)
ax.set_title('N-Step Return Spectrum: TD(0) to Monte Carlo', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 7. N-Step TD Prediction

In [ ]:
def n_step_td_prediction(
    env: RandomWalkEnv,
    n: int,
    n_episodes: int = 300,
    alpha: float = 0.1,
    gamma: float = 0.99
) -> Tuple[np.ndarray, List[float]]:
    """N-step TD prediction for estimating state values.

    Implements the n-step TD prediction algorithm which updates values
    using the n-step return: G_t^(n) = sum of discounted rewards + gamma^n * V(S_{t+n}).

    Args:
        env: The RandomWalkEnv environment.
        n: Number of steps for the n-step return.
        n_episodes: Number of episodes to run.
        alpha: Step-size parameter.
        gamma: Discount factor.

    Returns:
        V: Estimated value function (array of size n_states+2).
        rmse_history: RMSE vs true values at each episode.
    """
    n_states = env.n_states
    V = np.zeros(n_states + 2)  # include terminal states
    true_v = env.true_values(gamma)
    rmse_history = []

    for episode in range(n_episodes):
        state = env.reset()
        # Storage for states and rewards
        states = [state]
        rewards = [0.0]  # R_0 is not used, placeholder

        T = float('inf')  # terminal time
        t = 0

        while True:
            if t < T:
                # Take action (random policy: 50/50 left/right)
                action = np.random.randint(2)
                next_state, reward, done = env.step(action)
                states.append(next_state)
                rewards.append(reward)

                if done:
                    T = t + 1

            # Time step to update
            tau = t - n + 1

            if tau >= 0:
                # Compute n-step return G
                G = 0.0
                for i in range(tau + 1, min(tau + n, int(T)) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]

                # Bootstrap if we haven't reached terminal
                if tau + n < T:
                    G += (gamma ** n) * V[states[tau + n]]

                # Update
                s_tau = states[tau]
                if 1 <= s_tau <= n_states:  # only update non-terminal states
                    V[s_tau] += alpha * (G - V[s_tau])

            if tau == T - 1:
                break
            t += 1

        # Compute RMSE over non-terminal states
        rmse = np.sqrt(np.mean((V[1:n_states + 1] - true_v[1:n_states + 1]) ** 2))
        rmse_history.append(rmse)

    return V, rmse_history


print("n_step_td_prediction() defined.")

---
## 8. N-Step SARSA (Control)

In [ ]:
def n_step_sarsa(
    env: RandomWalkEnv,
    n: int,
    n_episodes: int = 300,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1
) -> Tuple[np.ndarray, List[float]]:
    """N-step SARSA for control.

    Uses epsilon-greedy policy derived from Q to select actions.
    Updates Q using n-step returns.

    Args:
        env: The RandomWalkEnv environment.
        n: Number of steps for the n-step return.
        n_episodes: Number of episodes to run.
        alpha: Step-size parameter.
        gamma: Discount factor.
        epsilon: Exploration rate for epsilon-greedy.

    Returns:
        Q: Estimated action-value function (n_states+2 x n_actions).
        episode_rewards: Total reward per episode.
    """
    n_states = env.n_states
    Q = np.zeros((n_states + 2, env.n_actions))
    episode_rewards = []

    def epsilon_greedy(state: int) -> int:
        """Select action using epsilon-greedy policy."""
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return int(np.argmax(Q[state]))

    for episode in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(state)

        states = [state]
        actions = [action]
        rewards = [0.0]

        T = float('inf')
        t = 0
        total_reward = 0.0

        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                total_reward += reward
                states.append(next_state)
                rewards.append(reward)

                if done:
                    T = t + 1
                    actions.append(0)  # dummy action for terminal
                else:
                    next_action = epsilon_greedy(next_state)
                    actions.append(next_action)

            tau = t - n + 1

            if tau >= 0:
                G = 0.0
                for i in range(tau + 1, min(tau + n, int(T)) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]

                if tau + n < T:
                    G += (gamma ** n) * Q[states[tau + n], actions[tau + n]]

                s_tau = states[tau]
                a_tau = actions[tau]
                if 1 <= s_tau <= n_states:
                    Q[s_tau, a_tau] += alpha * (G - Q[s_tau, a_tau])

            if tau == T - 1:
                break
            t += 1

        episode_rewards.append(total_reward)

    return Q, episode_rewards


print("n_step_sarsa() defined.")

---
## 9. TD($\lambda$) Prediction (Backward View with Eligibility Traces)

In [ ]:
def td_lambda_prediction(
    env: RandomWalkEnv,
    lam: float,
    n_episodes: int = 300,
    alpha: float = 0.1,
    gamma: float = 0.99,
    trace_type: str = 'accumulating'
) -> Tuple[np.ndarray, List[float]]:
    """TD(lambda) prediction using the backward view with eligibility traces.

    Implements the backward-view TD(lambda) algorithm. At each step,
    the TD error is propagated to all previously visited states according
    to their eligibility trace.

    Args:
        env: The RandomWalkEnv environment.
        lam: Lambda parameter (0 = TD(0), 1 = MC-like).
        n_episodes: Number of episodes to run.
        alpha: Step-size parameter.
        gamma: Discount factor.
        trace_type: 'accumulating' or 'replacing'.

    Returns:
        V: Estimated value function.
        rmse_history: RMSE vs true values at each episode.
    """
    n_states = env.n_states
    V = np.zeros(n_states + 2)
    true_v = env.true_values(gamma)
    rmse_history = []

    for episode in range(n_episodes):
        state = env.reset()
        # Initialize eligibility traces to zero
        e = np.zeros(n_states + 2)
        done = False

        while not done:
            # Random policy: 50/50 left/right
            action = np.random.randint(2)
            next_state, reward, done = env.step(action)

            # TD error
            delta = reward + gamma * V[next_state] * (1 - done) - V[state]

            # Update eligibility trace for current state
            if trace_type == 'accumulating':
                e *= gamma * lam
                e[state] += 1.0
            elif trace_type == 'replacing':
                e *= gamma * lam
                e[state] = max(e[state], 1.0)

            # Update all state values using traces
            V[1:n_states + 1] += alpha * delta * e[1:n_states + 1]

            state = next_state

        # Compute RMSE
        rmse = np.sqrt(np.mean((V[1:n_states + 1] - true_v[1:n_states + 1]) ** 2))
        rmse_history.append(rmse)

    return V, rmse_history


print("td_lambda_prediction() defined.")

---
## 10. SARSA($\lambda$) Control

In [ ]:
def td_lambda_control(
    env: RandomWalkEnv,
    lam: float,
    n_episodes: int = 300,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
    trace_type: str = 'accumulating'
) -> Tuple[np.ndarray, List[float]]:
    """SARSA(lambda) control with eligibility traces.

    Combines SARSA updates with eligibility traces for efficient
    credit assignment across multiple time steps.

    Args:
        env: The RandomWalkEnv environment.
        lam: Lambda parameter.
        n_episodes: Number of episodes to run.
        alpha: Step-size parameter.
        gamma: Discount factor.
        epsilon: Exploration rate.
        trace_type: 'accumulating' or 'replacing'.

    Returns:
        Q: Estimated action-value function.
        episode_rewards: Total reward per episode.
    """
    n_states = env.n_states
    Q = np.zeros((n_states + 2, env.n_actions))
    episode_rewards = []

    def epsilon_greedy(state: int) -> int:
        """Select action using epsilon-greedy policy."""
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return int(np.argmax(Q[state]))

    for episode in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(state)
        # Initialize eligibility traces
        e = np.zeros((n_states + 2, env.n_actions))
        done = False
        total_reward = 0.0

        while not done:
            next_state, reward, done = env.step(action)
            total_reward += reward

            if done:
                next_action = 0  # dummy
                delta = reward - Q[state, action]
            else:
                next_action = epsilon_greedy(next_state)
                delta = reward + gamma * Q[next_state, next_action] - Q[state, action]

            # Update eligibility trace
            if trace_type == 'accumulating':
                e *= gamma * lam
                e[state, action] += 1.0
            elif trace_type == 'replacing':
                e *= gamma * lam
                e[state, action] = max(e[state, action], 1.0)

            # Update all Q values
            Q[1:n_states + 1, :] += alpha * delta * e[1:n_states + 1, :]

            state = next_state
            action = next_action

        episode_rewards.append(total_reward)

    return Q, episode_rewards


print("td_lambda_control() defined.")

---
## 11. Experiment: Effect of n on Learning (RMSE vs Episodes)

In [ ]:
# Run n-step TD prediction for different values of n
np.random.seed(SEED)
n_values = [1, 2, 4, 8, 16]
results_n_step = {}

for n_val in n_values:
    env = RandomWalkEnv()
    np.random.seed(SEED)  # reset seed for fair comparison
    V, rmse_hist = n_step_td_prediction(env, n=n_val, n_episodes=N_EPISODES,
                                         alpha=ALPHA, gamma=GAMMA)
    results_n_step[n_val] = {'V': V, 'rmse': rmse_hist}
    print(f"n={n_val:>2d}: Final RMSE = {rmse_hist[-1]:.4f}")

# Plot RMSE vs episodes for different n
fig, ax = plt.subplots(figsize=(12, 5))
for idx, n_val in enumerate(n_values):
    # Smooth RMSE for readability
    rmse = results_n_step[n_val]['rmse']
    window = 10
    smoothed = np.convolve(rmse, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=COLORS[idx % len(COLORS)], label=f'n={n_val}')

ax.set_xlabel('Episode')
ax.set_ylabel('RMSE')
ax.set_title('Effect of n on N-Step TD Prediction (RMSE vs Episodes)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 12. Experiment: Effect of $\lambda$ on TD($\lambda$)

In [ ]:
# Run TD(lambda) for different values of lambda
lambda_values = [0.0, 0.4, 0.8, 0.95, 1.0]
results_lambda = {}

for lam_val in lambda_values:
    env = RandomWalkEnv()
    np.random.seed(SEED)
    V, rmse_hist = td_lambda_prediction(env, lam=lam_val, n_episodes=N_EPISODES,
                                         alpha=ALPHA, gamma=GAMMA)
    results_lambda[lam_val] = {'V': V, 'rmse': rmse_hist}
    print(f"lambda={lam_val:.2f}: Final RMSE = {rmse_hist[-1]:.4f}")

# Plot RMSE vs episodes for different lambda
fig, ax = plt.subplots(figsize=(12, 5))
for idx, lam_val in enumerate(lambda_values):
    rmse = results_lambda[lam_val]['rmse']
    window = 10
    smoothed = np.convolve(rmse, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=COLORS[idx % len(COLORS)], label=f'$\\lambda$={lam_val}')

ax.set_xlabel('Episode')
ax.set_ylabel('RMSE')
ax.set_title('Effect of $\\lambda$ on TD($\\lambda$) Prediction (RMSE vs Episodes)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 13. Eligibility Trace Decay Visualization

In [ ]:
def visualize_eligibility_traces(gamma: float = 0.99, lam: float = 0.8, n_steps: int = 30):
    """Visualize how eligibility traces evolve over time steps.

    Simulates a random walk and records the eligibility trace of each
    visited state across time steps.

    Args:
        gamma: Discount factor.
        lam: Lambda parameter.
        n_steps: Number of steps to simulate.
    """
    env = RandomWalkEnv()
    np.random.seed(SEED)
    state = env.reset()

    n_states = env.n_states
    e = np.zeros(n_states + 2)
    trace_history = []
    visited_states = set()

    for step in range(n_steps):
        action = np.random.randint(2)
        next_state, reward, done = env.step(action)

        # Update trace (accumulating)
        e *= gamma * lam
        e[state] += 1.0
        visited_states.add(state)

        trace_history.append(e.copy())

        if done:
            break
        state = next_state

    trace_history = np.array(trace_history)

    # Plot traces for visited states
    visited = sorted(visited_states)
    fig, ax = plt.subplots(figsize=(12, 5))

    # Select a subset of visited states to show (at most 8 for clarity)
    show_states = visited[:8] if len(visited) > 8 else visited
    color_cycle = plt.cm.tab10(np.linspace(0, 1, len(show_states)))

    for idx, s in enumerate(show_states):
        ax.plot(trace_history[:, s], color=color_cycle[idx],
                label=f'State {s}', linewidth=2, alpha=0.8)

    ax.set_xlabel('Time Step')
    ax.set_ylabel('Eligibility Trace Value')
    ax.set_title(f'Eligibility Trace Decay ($\\gamma$={gamma}, $\\lambda$={lam})',
                 fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


visualize_eligibility_traces(gamma=GAMMA, lam=0.8, n_steps=30)

The figure above shows how eligibility traces rise when a state is visited and then decay exponentially by a factor of $\gamma\lambda$ per time step. States that are visited more recently or more frequently maintain higher traces, receiving larger credit from the TD error.

---
## 14. Learned Values vs True Values

In [ ]:
env = RandomWalkEnv()
true_v = env.true_values(gamma=GAMMA)
states = np.arange(1, env.n_states + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Different n values
ax = axes[0]
ax.plot(states, true_v[1:-1], 'k--', linewidth=2.5, label='True Values', alpha=0.7)
for idx, n_val in enumerate([1, 4, 16]):
    V = results_n_step[n_val]['V']
    ax.plot(states, V[1:-1], color=COLORS[idx], marker='o', markersize=4,
            label=f'n={n_val}')
ax.set_xlabel('State')
ax.set_ylabel('Value')
ax.set_title('N-Step TD: Learned vs True Values', fontweight='bold')
ax.legend()

# Right panel: Different lambda values
ax = axes[1]
ax.plot(states, true_v[1:-1], 'k--', linewidth=2.5, label='True Values', alpha=0.7)
for idx, lam_val in enumerate([0.0, 0.8, 1.0]):
    V = results_lambda[lam_val]['V']
    ax.plot(states, V[1:-1], color=COLORS[idx], marker='o', markersize=4,
            label=f'$\\lambda$={lam_val}')
ax.set_xlabel('State')
ax.set_ylabel('Value')
ax.set_title('TD($\\lambda$): Learned vs True Values', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

---
## 15. Alpha Sensitivity: Parameter Study

In [ ]:
# Parameter study: average RMSE over last 50 episodes vs alpha for different n
alpha_values = np.linspace(0.01, 0.5, 15)
n_test_values = [1, 2, 4, 8, 16]
avg_rmse_results = {n_val: [] for n_val in n_test_values}

for n_val in n_test_values:
    for alpha_val in alpha_values:
        env = RandomWalkEnv()
        np.random.seed(SEED)
        _, rmse_hist = n_step_td_prediction(env, n=n_val, n_episodes=150,
                                             alpha=alpha_val, gamma=GAMMA)
        # Average RMSE over last 50 episodes
        avg_rmse = np.mean(rmse_hist[-50:])
        avg_rmse_results[n_val].append(avg_rmse)

fig, ax = plt.subplots(figsize=(12, 5))
for idx, n_val in enumerate(n_test_values):
    ax.plot(alpha_values, avg_rmse_results[n_val], color=COLORS[idx % len(COLORS)],
            marker='o', markersize=5, label=f'n={n_val}')

ax.set_xlabel('$\\alpha$ (Step Size)')
ax.set_ylabel('Average RMSE (last 50 episodes)')
ax.set_title('$\\alpha$ Sensitivity for N-Step TD Prediction', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Find best alpha for each n
for n_val in n_test_values:
    best_idx = np.argmin(avg_rmse_results[n_val])
    print(f"n={n_val:>2d}: best alpha = {alpha_values[best_idx]:.3f}, "
          f"RMSE = {avg_rmse_results[n_val][best_idx]:.4f}")

---
## 16. N-Step SARSA: Learning Curves

In [ ]:
# Run n-step SARSA for control
np.random.seed(SEED)
n_sarsa_values = [1, 4, 8]
sarsa_results = {}

for n_val in n_sarsa_values:
    env = RandomWalkEnv()
    np.random.seed(SEED)
    Q, ep_rewards = n_step_sarsa(env, n=n_val, n_episodes=N_EPISODES,
                                  alpha=ALPHA, gamma=GAMMA, epsilon=0.1)
    sarsa_results[n_val] = {'Q': Q, 'rewards': ep_rewards}
    avg_last = np.mean(ep_rewards[-50:])
    print(f"n-step SARSA (n={n_val}): avg reward (last 50 ep) = {avg_last:.3f}")

# Plot learning curves
fig, ax = plt.subplots(figsize=(12, 5))
window = 20
for idx, n_val in enumerate(n_sarsa_values):
    rewards = sarsa_results[n_val]['rewards']
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=COLORS[idx], label=f'n-step SARSA (n={n_val})')

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('N-Step SARSA: Learning Curves', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 17. SARSA($\lambda$): Learning Curves

In [ ]:
# Run SARSA(lambda) for different lambda values
lambda_ctrl_values = [0.0, 0.4, 0.8, 0.95]
sarsa_lambda_results = {}

for lam_val in lambda_ctrl_values:
    env = RandomWalkEnv()
    np.random.seed(SEED)
    Q, ep_rewards = td_lambda_control(env, lam=lam_val, n_episodes=N_EPISODES,
                                       alpha=ALPHA, gamma=GAMMA, epsilon=0.1)
    sarsa_lambda_results[lam_val] = {'Q': Q, 'rewards': ep_rewards}
    avg_last = np.mean(ep_rewards[-50:])
    print(f"SARSA(lambda={lam_val:.2f}): avg reward (last 50 ep) = {avg_last:.3f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
window = 20
for idx, lam_val in enumerate(lambda_ctrl_values):
    rewards = sarsa_lambda_results[lam_val]['rewards']
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=COLORS[idx], label=f'SARSA($\\lambda$={lam_val})')

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('SARSA($\\lambda$): Effect of $\\lambda$ on Control', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 18. Accumulating vs Replacing Traces

In [ ]:
# Compare accumulating vs replacing traces for TD(lambda)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lam_test = 0.8

for idx, trace_type in enumerate(['accumulating', 'replacing']):
    env = RandomWalkEnv()
    np.random.seed(SEED)
    V, rmse_hist = td_lambda_prediction(env, lam=lam_test, n_episodes=N_EPISODES,
                                         alpha=ALPHA, gamma=GAMMA,
                                         trace_type=trace_type)
    window = 10
    smoothed = np.convolve(rmse_hist, np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed, color=COLORS[idx], label=f'{trace_type.capitalize()}')

    # Plot learned vs true values
    true_v = env.true_values(gamma=GAMMA)
    states = np.arange(1, env.n_states + 1)
    axes[1].plot(states, V[1:-1], color=COLORS[idx], marker='o', markersize=4,
                 label=f'{trace_type.capitalize()}')

axes[1].plot(states, true_v[1:-1], 'k--', linewidth=2.5, label='True Values', alpha=0.7)

axes[0].set_xlabel('Episode')
axes[0].set_ylabel('RMSE')
axes[0].set_title(f'RMSE: Accumulating vs Replacing ($\\lambda$={lam_test})', fontweight='bold')
axes[0].legend()

axes[1].set_xlabel('State')
axes[1].set_ylabel('Value')
axes[1].set_title(f'Learned Values ($\\lambda$={lam_test})', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 19. Verification Tests

In [ ]:
def run_verification_tests():
    """Run a battery of verification tests comparing methods.

    Tests:
    1. N-step TD (n=1) matches TD(0) results.
    2. TD(lambda=0) matches TD(0).
    3. TD(lambda=1) approximates MC (with higher variance).
    4. RMSE decreases over episodes.
    5. Intermediate n outperforms extremes.

    Returns:
        None (prints results).
    """
    print("=" * 60)
    print("VERIFICATION TESTS")
    print("=" * 60)

    env = RandomWalkEnv()
    true_v = env.true_values(gamma=GAMMA)

    # --- Test 1: n-step TD(n=1) matches TD(0) ---
    # Both should produce the same result with the same seed
    np.random.seed(SEED)
    V_n1, rmse_n1 = n_step_td_prediction(env, n=1, n_episodes=200, alpha=ALPHA, gamma=GAMMA)

    np.random.seed(SEED)
    V_td0, rmse_td0 = td_lambda_prediction(env, lam=0.0, n_episodes=200, alpha=ALPHA, gamma=GAMMA)

    diff_n1_td0 = np.max(np.abs(V_n1[1:-1] - V_td0[1:-1]))
    test1 = diff_n1_td0 < 0.15  # small tolerance due to implementation differences
    print(f"\nTest 1: n-step TD(n=1) ~ TD(0)")
    print(f"  Max value difference: {diff_n1_td0:.4f}")
    print(f"  Result: {'[PASS]' if test1 else '[FAIL]'}")

    # --- Test 2: TD(lambda=0) matches TD(0) behavior ---
    np.random.seed(SEED)
    V_lam0, rmse_lam0 = td_lambda_prediction(env, lam=0.0, n_episodes=200,
                                               alpha=ALPHA, gamma=GAMMA)
    # TD(0) should only use 1-step returns; with lambda=0 traces decay instantly
    rmse_final_lam0 = rmse_lam0[-1]
    rmse_final_n1 = rmse_n1[-1]
    test2 = abs(rmse_final_lam0 - rmse_final_n1) < 0.15
    print(f"\nTest 2: TD(lambda=0) ~ TD(0)")
    print(f"  Final RMSE — TD(lambda=0): {rmse_final_lam0:.4f}, n-step(n=1): {rmse_final_n1:.4f}")
    print(f"  Result: {'[PASS]' if test2 else '[FAIL]'}")

    # --- Test 3: TD(lambda=1) approximates MC (higher variance expected) ---
    np.random.seed(SEED)
    V_lam1, rmse_lam1 = td_lambda_prediction(env, lam=1.0, n_episodes=200,
                                               alpha=ALPHA, gamma=GAMMA)
    # TD(lambda=1) should eventually converge but with higher variance
    rmse_early_lam1 = np.mean(rmse_lam1[:20])
    rmse_early_lam0 = np.mean(rmse_lam0[:20])
    # Lambda=1 typically has higher early RMSE variance
    test3 = rmse_lam1[-1] < 0.5  # still converges reasonably
    print(f"\nTest 3: TD(lambda=1) approximates MC")
    print(f"  Final RMSE TD(lambda=1): {rmse_lam1[-1]:.4f}")
    print(f"  Early RMSE — lambda=0: {rmse_early_lam0:.4f}, lambda=1: {rmse_early_lam1:.4f}")
    print(f"  (lambda=1 should show higher variance, MC-like behavior)")
    print(f"  Result: {'[PASS]' if test3 else '[FAIL]'}")

    # --- Test 4: RMSE decreases over episodes ---
    np.random.seed(SEED)
    _, rmse_test = n_step_td_prediction(env, n=4, n_episodes=300, alpha=ALPHA, gamma=GAMMA)
    first_quarter = np.mean(rmse_test[:75])
    last_quarter = np.mean(rmse_test[-75:])
    test4 = last_quarter < first_quarter
    print(f"\nTest 4: RMSE decreases over episodes (n=4)")
    print(f"  Avg RMSE first quarter: {first_quarter:.4f}")
    print(f"  Avg RMSE last quarter:  {last_quarter:.4f}")
    print(f"  Result: {'[PASS]' if test4 else '[FAIL]'}")

    # --- Test 5: Intermediate n outperforms extremes ---
    final_rmses = {}
    for n_val in [1, 4, 8, 16]:
        np.random.seed(SEED)
        _, rmse_h = n_step_td_prediction(env, n=n_val, n_episodes=300,
                                          alpha=ALPHA, gamma=GAMMA)
        final_rmses[n_val] = np.mean(rmse_h[-50:])

    best_n = min(final_rmses, key=final_rmses.get)
    test5 = best_n not in [1]  # intermediate n should be competitive or best
    print(f"\nTest 5: Intermediate n outperforms extremes")
    for n_val, rmse_val in sorted(final_rmses.items()):
        marker = " <-- best" if n_val == best_n else ""
        print(f"  n={n_val:>2d}: avg RMSE (last 50 ep) = {rmse_val:.4f}{marker}")
    print(f"  Best n = {best_n}")
    print(f"  Result: {'[PASS]' if test5 else '[FAIL]'}")

    # --- Summary ---
    all_tests = [test1, test2, test3, test4, test5]
    passed = sum(all_tests)
    print(f"\n{'=' * 60}")
    print(f"SUMMARY: {passed}/{len(all_tests)} tests passed")
    print(f"{'=' * 60}")


run_verification_tests()

---
## 20. Summary and Key Takeaways

### N-Step Methods
- **N-step returns** unify TD(0) and Monte Carlo on a spectrum controlled by $n$
- Small $n$ gives low variance but high bias; large $n$ gives low bias but high variance
- **Intermediate values of $n$** (e.g., 4 or 8) often perform best, balancing bias and variance
- Implementation requires storing $n$ transitions before making updates

### Eligibility Traces
- Traces elegantly combine **recency** and **frequency** heuristics for credit assignment
- **Accumulating traces** can grow unboundedly; **replacing traces** cap at 1
- Traces allow efficient online updates that propagate information backward through time

### TD($\lambda$)
- The $\lambda$-return is a **geometrically weighted average** of all n-step returns
- **Forward view** (conceptual): compute $G_t^\lambda$ explicitly
- **Backward view** (practical): use eligibility traces for incremental, online updates
- $\lambda = 0$ recovers TD(0); $\lambda = 1$ recovers Monte Carlo (offline case)
- **Intermediate $\lambda$** values (0.4–0.9) typically yield the best performance

### Practical Considerations
- The **step size $\alpha$** must be tuned alongside $n$ or $\lambda$ — larger $n$ often requires smaller $\alpha$
- Eligibility traces add minimal computational overhead (one vector of trace values)
- These methods scale well to function approximation (linear and neural networks)